<a href="https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:

import os
import sys
import subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "duckdb",
            "huggingface_hub",
            "pandas",
            "scikit-learn"
        ],
        check=True
    )

elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

con = duckdb.connect()

con.execute(
    f"""
    CREATE SECRET (
        TYPE huggingface,
        TOKEN '{userdata.get("HF_TOKEN")}'
    )
    """
)

rel = "hf://datasets/FlyRank/internship-warehouse"

df = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        AVG(gsc_avg_position) AS avg_position,

        LN(SUM(gsc_impressions) + 1) AS log_impressions,

        SUM(ga4_engaged_sessions) AS engaged_sessions,

        SUM(ga4_total_engagement_sec) AS total_engagement_sec,

        SUM(scroll_events) AS scroll_events,

        SUM(gsc_clicks) AS total_clicks,

        SUM(gsc_impressions) AS total_impressions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) > 0
    """
).df()


df["ctr"] = (
    df["total_clicks"] /
    df["total_impressions"]
)

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 50, 1000],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"]
)

bucket_median_ctr = (
    df.groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .transform("median")
)

df["is_underperforming"] = (
    df["ctr"] <= bucket_median_ctr
).astype(int)

honest_features = [
    "avg_position",
    "log_impressions",
    "engaged_sessions",
    "total_engagement_sec",
    "scroll_events"
]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df["client_hash_id"]
    )
)

train_df = (
    df.iloc[train_idx]
    .reset_index(drop=True)
)

test_df = (
    df.iloc[test_idx]
    .reset_index(drop=True)
)

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(
    train_df[honest_features].fillna(0),
    train_df["is_underperforming"]
)

test_df["model_score"] = (
    model.predict_proba(
        test_df[honest_features].fillna(0)
    )[:, 1]
)

median_by_bucket = (
    train_df
    .groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .median()
)

test_df["bucket_median_ctr"] = (
    test_df["position_bucket"]
    .map(median_by_bucket)
)


def reason_code(row):

    # Strong position + zero clicks
    if (
        row["ctr"] == 0
        and row["avg_position"] <= 10
    ):
        return "ZERO_CLICKS_WITH_STRONG_POSITION"

    # CTR genuinely below position peers
    elif (
        row["ctr"] < row["bucket_median_ctr"]
    ):
        return "CTR_BELOW_POSITION_PEERS"

    # Zero CTR but peers also have zero CTR
    elif (
        row["ctr"] == 0
        and row["bucket_median_ctr"] == 0
    ):
        return "ZERO_CTR_WITH_ZERO_PEER_BASELINE"

    # No strong signal
    else:
        return "NO_STRONG_SIGNAL"


test_df["reason_code"] = test_df.apply(
    reason_code,
    axis=1
)


ranked_queue = (
    test_df[
        test_df["is_underperforming"] == 1
    ]
    .sort_values(
        "model_score",
        ascending=False
    )
    .reset_index(drop=True)
)

ranked_queue["rank"] = (
    ranked_queue.index + 1
)


ranked_queue["decay_signal"] = (
    (ranked_queue["engaged_sessions"] < ranked_queue["engaged_sessions"].median()) &
    (ranked_queue["total_engagement_sec"] < ranked_queue["total_engagement_sec"].median())
)

def assign_action(row):

    # Strong position but zero clicks
    if (
        row["reason_code"]
        == "ZERO_CLICKS_WITH_STRONG_POSITION"
    ):
        return "REVIEW_SNIPPET_TITLE"

    # Engagement decay
    elif row["decay_signal"]:
        return "CANDIDATE_FOR_REFRESH"

    # CTR below peers
    elif (
        row["reason_code"]
        == "CTR_BELOW_POSITION_PEERS"
    ):
        return "REVIEW_RELEVANCE_INTENT_MATCH"

    # Weak evidence
    else:
        return "MONITOR"


ranked_queue["recommended_action"] = (
    ranked_queue.apply(
        assign_action,
        axis=1
    )
)

display(
    ranked_queue[
        [
            "rank",
            "content_hash_id",
            "avg_position",
            "ctr",
            "bucket_median_ctr",
            "model_score",
            "reason_code",
            "decay_signal",
            "recommended_action"
        ]
    ].head(15)
)
print("\nAction distribution:")

print(
    ranked_queue[
        "recommended_action"
    ].value_counts()
)

print("\nReason-code distribution:")

print(
    ranked_queue[
        "reason_code"
    ].value_counts()
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rank,content_hash_id,avg_position,ctr,bucket_median_ctr,model_score,reason_code,decay_signal,recommended_action
0,1,content_fc468c5940d16ea3,262.0,0.0,0.0,1.000000,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
1,2,content_31e042d90d0176c6,215.0,0.0,0.0,0.999999,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
2,3,content_e8f3910a442bf204,191.0,0.0,0.0,0.999996,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
3,4,content_fef453db040622b6,167.0,0.0,0.0,0.999989,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
4,5,content_fc1c5c2867b3f36f,173.5,0.0,0.0,0.999989,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
5,6,content_67a38ea5a62cca5e,163.0,0.0,0.0,0.999986,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
6,7,content_376352e250b80228,156.0,0.0,0.0,0.999981,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
7,8,content_1e6be49007ce0769,155.0,0.0,0.0,0.999980,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
8,9,content_6db90d1ad60b4644,153.0,0.0,0.0,0.999978,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR
9,10,content_652837b23594c953,159.0,0.0,0.0,0.999978,ZERO_CTR_WITH_ZERO_PEER_BASELINE,False,MONITOR



Action distribution:
recommended_action
MONITOR                 14952
REVIEW_SNIPPET_TITLE    10555
Name: count, dtype: int64

Reason-code distribution:
reason_code
ZERO_CTR_WITH_ZERO_PEER_BASELINE    14952
ZERO_CLICKS_WITH_STRONG_POSITION    10555
Name: count, dtype: int64


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended Use and Limits

### Intended use

This action queue is intended to support human review of content performance.

The model ranks pages by their predicted likelihood of underperformance, while the reason codes provide a simple explanation for why a page was flagged. The queue helps an SEO or content team prioritize which pages should be reviewed first.

The output is decision-support, not an automatic recommendation to change a page.

### Limits

The model does not prove that a page needs a specific change. A recommended action such as `REVIEW_SNIPPET_TITLE` means that the page is a candidate for human review, not that the title or snippet is definitely the cause of the problem.

The model also does not:

- predict Google's future rankings;
- establish that one change will cause better performance;
- replace SEO or content expertise;
- automatically rewrite or publish content;
- guarantee that a recommended action will improve performance.

The ranked queue should therefore be treated as a prioritization tool. Final decisions remain with a human reviewer who can consider additional context that is not included in the model.


A note on reason-code coverage: in this test-set population, CTR_BELOW_POSITION_PEERS and CANDIDATE_FOR_REFRESH never populate. Both engaged_sessions and total_engagement_sec sit at their own median value of 0 across this underperforming set, so a strict "below median" comparison can never be true — the comparison is mathematically unreachable, not evidence of an absence of decay. Loosening the comparison to "at or below median" was tested and instead over-flagged the large majority of pages as refresh candidates, which is equally uninformative. This reflects a genuine floor effect in the underlying data (most zero-CTR pages also show zero engagement), and is a limitation of this feature set rather than a tuned threshold choice. A future iteration could use a different signal (e.g., days since last update, if available) to distinguish decay from simple low-traffic pages.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human Review + No-Go List

### Human review rules

The ranked queue is a prioritization tool. Every recommended action should be reviewed by a human before any change is made.

For each page, the reviewer should:

1. Check the page's current search position and CTR.
2. Review the reason code and confirm that it matches the available evidence.
3. Check the search intent and whether the content matches what users are searching for.
4. Review the current title and snippet when the action is `REVIEW_SNIPPET_TITLE`.
5. Look for additional context that is not included in the model features.
6. Decide whether to act, monitor, or reject the recommendation.

A high model score means higher priority for review. It does not mean that the recommended action is definitely correct.

### No-go list: what should NOT be automated

The following decisions should not be automatically executed:

- Automatically changing or publishing page titles.
- Automatically rewriting or publishing content.
- Automatically changing search-targeting or user intent.
- Automatically deleting or redirecting pages.
- Automatically making SEO changes based only on the model score.
- Treating the model output as proof of causation.
- Automatically declaring a page successful or unsuccessful after a change.

These actions can affect real users, search visibility, and business outcomes, so they require human judgment and review.

The model should prioritize and provide decision-support, while humans remain responsible for the final action.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / Retrain Triggers

The model should be monitored over time rather than assumed to remain valid indefinitely.

### What to monitor

We should periodically check:

- Precision@K or another validation metric on new data.
- The distribution of model scores.
- The distribution of reason codes and recommended actions.
- Whether the data contains unexpected missing values or changes in feature distributions.
- Whether the types of pages being flagged change substantially.

### Possible retrain triggers

Retraining should be considered when:

- validation performance drops meaningfully on newer data;
- the underlying data or feature distributions change substantially;
- the relationship between the features and underperformance appears to change;
- new data or content types are introduced that were not represented in the training data.

Retraining should not happen simply because a scheduled date has arrived. There should be evidence that the model or its data has changed enough to justify it.

### Monitoring principle

The action queue is decision-support, so monitoring should focus on whether it continues to provide useful and trustworthy prioritization.

If performance or data quality deteriorates, the model should be reviewed before its output is used for further decisions.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [2]:
# ============================================================
# 5. Exports for the Paper
# ============================================================

import os
import json

# Create output folders if they do not already exist
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)


# ------------------------------------------------------------
# Export the ranked action queue
# ------------------------------------------------------------

queue_path = "work/outputs/w07_ranked_action_queue.csv"

ranked_queue[
    [
        "rank",
        "content_hash_id",
        "avg_position",
        "ctr",
        "bucket_median_ctr",
        "model_score",
        "reason_code",
        "decay_signal",
        "recommended_action"
    ]
].to_csv(
    queue_path,
    index=False
)

print(f"Ranked queue exported to: {queue_path}")


# ------------------------------------------------------------
# Export summary metrics
# ------------------------------------------------------------

metrics = {
    "total_ranked_pages": int(len(ranked_queue)),
    "monitor_pages": int(
        (ranked_queue["recommended_action"] == "MONITOR").sum()
    ),
    "snippet_review_pages": int(
        (
            ranked_queue["recommended_action"]
            == "REVIEW_SNIPPET_TITLE"
        ).sum()
    ),
    "refresh_candidates": int(
        (
            ranked_queue["recommended_action"]
            == "CANDIDATE_FOR_REFRESH"
        ).sum()
    ),
    "intent_review_candidates": int(
        (
            ranked_queue["recommended_action"]
            == "REVIEW_RELEVANCE_INTENT_MATCH"
        ).sum()
    )
}

metrics_path = "work/outputs/w07_action_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics exported to: {metrics_path}")

print("\nMetrics:")
print(json.dumps(metrics, indent=2))

Ranked queue exported to: work/outputs/w07_ranked_action_queue.csv
Metrics exported to: work/outputs/w07_action_metrics.json

Metrics:
{
  "total_ranked_pages": 25507,
  "monitor_pages": 14952,
  "snippet_review_pages": 10555,
  "refresh_candidates": 0,
  "intent_review_candidates": 0
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.


Before you submit, confirm each line honestly:

- ☑ Every section above is filled — markdown thinking AND the code that backs it
- ☑ The notebook runs top to bottom with no errors (Runtime → Run all)
- ☑ No client names, URLs, or private queries anywhere
- ☑ My claims use careful words: observed, measured, directional, decision-support
- ☑ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.